In [ ]:
workspace = "urban-analytics"
window_start = "2024-01-01"
window_end = "2024-06-30"
sources = ""


In [ ]:
from datetime import date

from urban_analytics.platform import Platform, Window

platform = Platform.fabric(workspace, Window(date.fromisoformat(window_start), date.fromisoformat(window_end)))


In [ ]:
from urban_analytics.silver import build_silver

only = tuple(name for name in sources.split(",") if name)
results = build_silver(spark, platform, only)
display(spark.createDataFrame([(r.source, r.accepted, r.rejected) for r in results], ["source", "accepted", "rejected"]))


In [ ]:
from urban_analytics.registry import registered_sources
from urban_analytics.silver import rejected_table

for name in registered_sources():
    rejected = spark.table(platform.silver.table(rejected_table(name)))
    if rejected.count():
        print(name)
        rejected.groupBy("reject_reason").count().orderBy("count", ascending=False).show(truncate=False)
